# 02.3 PagedAttention: Virtual Memory for KV Cache

PagedAttention applies OS-style virtual memory to KV cache management:
non-contiguous physical blocks mapped via a block table, eliminating external
fragmentation and enabling prefix sharing + copy-on-write.

**Experiments:**
1. Block allocator implementation & allocation patterns
2. Fragmentation analysis: contiguous vs paged
3. Block size sweep — finding the sweet spot
4. Prefix sharing with ref-counted blocks
5. Copy-on-write for beam search
6. Throughput under load simulation
7. End-to-end memory efficiency with real model params

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Optional
from utils.kv_efficiency import kv_cache_size_gib
from utils.gpu_info import get_gpu_info, GPUInfo

## 1. Block Allocator

The core data structure: a pool of fixed-size physical blocks with ref counting.
Logical-to-physical mapping is stored in a block table (one per sequence).

In [ ]:
@dataclass
class Block:
    block_id: int
    ref_count: int = 1
    tokens_filled: int = 0

class BlockAllocator:
    """Physical block pool with ref counting."""
    def __init__(self, num_blocks: int, block_size: int = 16):
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks - 1, -1, -1))  # stack
        self.blocks: Dict[int, Block] = {}

    @property
    def num_free(self): return len(self.free_blocks)

    @property
    def num_used(self): return len(self.blocks)

    def allocate(self) -> Optional[int]:
        if not self.free_blocks: return None
        bid = self.free_blocks.pop()
        self.blocks[bid] = Block(block_id=bid)
        return bid

    def free(self, block_id: int):
        if block_id not in self.blocks: return
        self.blocks[block_id].ref_count -= 1
        if self.blocks[block_id].ref_count <= 0:
            del self.blocks[block_id]
            self.free_blocks.append(block_id)

    def ref(self, block_id: int):
        """Increment ref count for sharing."""
        if block_id in self.blocks:
            self.blocks[block_id].ref_count += 1

    def cow_copy(self, block_id: int) -> Optional[int]:
        """Copy-on-write: duplicate block, decrement original ref."""
        new_bid = self.allocate()
        if new_bid is not None and block_id in self.blocks:
            self.blocks[new_bid].tokens_filled = self.blocks[block_id].tokens_filled
            self.blocks[block_id].ref_count -= 1
        return new_bid

print('BlockAllocator ready')

In [ ]:
# --- Allocate sequences of varying lengths ---
allocator = BlockAllocator(num_blocks=1024, block_size=16)
seq_lengths = [47, 128, 256, 512, 1024, 63, 200, 95]

allocations = {}
for i, seq_len in enumerate(seq_lengths):
    n_blocks = (seq_len + allocator.block_size - 1) // allocator.block_size
    blocks = [allocator.allocate() for _ in range(n_blocks)]
    allocations[i] = (seq_len, blocks, n_blocks)
    # Mark fill levels
    for b in blocks[:-1]:
        allocator.blocks[b].tokens_filled = allocator.block_size
    last_fill = seq_len % allocator.block_size or allocator.block_size
    allocator.blocks[blocks[-1]].tokens_filled = last_fill

print(f'Allocated {len(seq_lengths)} sequences:')
for i, (slen, blks, nb) in allocations.items():
    waste = nb * 16 - slen
    print(f'  seq_{i}: {slen:>5} tokens -> {nb:>3} blocks, internal waste: {waste:>2} tokens')
print(f'\nBlocks used: {allocator.num_used}/1024  Free: {allocator.num_free}')

## 2. Fragmentation: Contiguous vs Paged

Contiguous allocation reserves `max_seq_len` per request (external fragmentation).
Paged allocation reserves only needed blocks (internal fragmentation bounded by `block_size - 1`).

In [ ]:
def contiguous_alloc(seq_lengths, total_tokens, max_seq=2048):
    """Pre-allocate max_seq_len per request."""
    used, served, wasted = 0, 0, 0
    for sl in seq_lengths:
        if used + max_seq <= total_tokens:
            used += max_seq
            wasted += (max_seq - sl)
            served += 1
    return {'served': served, 'wasted': wasted, 'used': used}

def paged_alloc(seq_lengths, total_tokens, block_size=16):
    """Allocate only needed blocks."""
    total_blocks = total_tokens // block_size
    used_blocks, served, wasted = 0, 0, 0
    for sl in seq_lengths:
        needed = (sl + block_size - 1) // block_size
        if used_blocks + needed <= total_blocks:
            used_blocks += needed
            wasted += (needed * block_size - sl)
            served += 1
    return {'served': served, 'wasted': wasted, 'used': used_blocks * block_size}

# Realistic workload: 200 requests, lengths 32-2048
np.random.seed(42)
workload = np.random.randint(32, 2048, size=200).tolist()
total_mem = 2048 * 200  # token-slots

c = contiguous_alloc(workload, total_mem)
p = paged_alloc(workload, total_mem)

print('=== Fragmentation Comparison (200 requests, 32-2048 tokens) ===')
print(f'Contiguous: served {c["served"]}/200, wasted {c["wasted"]:,} tokens ({c["wasted"]/c["used"]:.0%} of allocated)')
print(f'Paged:      served {p["served"]}/200, wasted {p["wasted"]:,} tokens ({p["wasted"]/p["used"]:.0%} of allocated)')
print(f'\nPaged serves {p["served"] - c["served"]} MORE requests from same memory')

In [ ]:
# --- Block Size Sweep ---
block_sizes = [1, 4, 8, 16, 32, 64, 128, 256]
print(f'{"BS":>4} | {"Served":>6} | {"Waste%":>7} | {"Wasted Tokens":>14}')
print('-' * 48)
for bs in block_sizes:
    r = paged_alloc(workload, total_mem, block_size=bs)
    frag_pct = r['wasted'] / r['used'] * 100 if r['used'] else 0
    print(f'{bs:>4} | {r["served"]:>6} | {frag_pct:>6.2f}% | {r["wasted"]:>14,}')

print('\nBlock size 16: best trade-off (low fragmentation, efficient CUDA kernels)')

## 3. Block Table Management

Each sequence maintains a block table: `logical_block_idx -> physical_block_id`.
The GPU attention kernel uses this table to gather KV vectors from non-contiguous memory.

In [ ]:
class PagedKVCache:
    """Full paged KV cache with block tables, prefix sharing, and CoW."""
    def __init__(self, num_blocks: int, block_size: int = 16):
        self.alloc = BlockAllocator(num_blocks, block_size)
        self.bs = block_size
        self.block_tables: Dict[int, List[int]] = {}  # seq_id -> [physical_block_ids]

    def add_sequence(self, seq_id: int, num_tokens: int, share_prefix_from: Optional[int] = None, prefix_blocks: int = 0):
        blocks = []
        tokens_remaining = num_tokens
        if share_prefix_from is not None and share_prefix_from in self.block_tables:
            shared = self.block_tables[share_prefix_from][:prefix_blocks]
            for bid in shared:
                self.alloc.ref(bid)
                blocks.append(bid)
            tokens_remaining -= prefix_blocks * self.bs
        # Allocate unique suffix
        n_new = (max(0, tokens_remaining) + self.bs - 1) // self.bs
        for _ in range(n_new):
            bid = self.alloc.allocate()
            if bid is not None: blocks.append(bid)
        self.block_tables[seq_id] = blocks

    def free_sequence(self, seq_id: int):
        for bid in self.block_tables.pop(seq_id, []):
            self.alloc.free(bid)

    def logical_to_physical(self, seq_id: int, token_pos: int) -> tuple:
        """Returns (physical_block_id, offset_within_block)."""
        logical_idx = token_pos // self.bs
        offset = token_pos % self.bs
        return self.block_tables[seq_id][logical_idx], offset

# Demo: show block table mapping
cache = PagedKVCache(num_blocks=256, block_size=16)
cache.add_sequence(0, num_tokens=100)

print('Block table for seq_0 (100 tokens, 7 blocks):')
print(f'  Logical -> Physical: {list(enumerate(cache.block_tables[0]))}')
print(f'\nToken 50 maps to: block {cache.logical_to_physical(0, 50)}')
print(f'Token 99 maps to: block {cache.logical_to_physical(0, 99)}')

## 4. Prefix Sharing

Multiple requests with the same system prompt share physical blocks via ref counting.
Only the unique suffix gets new allocations.

In [ ]:
# 8 requests sharing a 256-token system prompt
cache2 = PagedKVCache(num_blocks=2048, block_size=16)
prefix_len = 256
prefix_blocks = prefix_len // 16  # 16 blocks

# First request allocates prefix + unique suffix
cache2.add_sequence(0, num_tokens=prefix_len + 150)

# Remaining 7 share the prefix
np.random.seed(7)
for i in range(1, 8):
    unique = np.random.randint(64, 300)
    cache2.add_sequence(i, num_tokens=prefix_len + unique,
                        share_prefix_from=0, prefix_blocks=prefix_blocks)

# Memory accounting
without_sharing = 8 * prefix_blocks  # each would allocate own prefix
with_sharing = prefix_blocks  # one physical copy
saved_blocks = without_sharing - with_sharing

print('=== Prefix Sharing (8 requests, 256-token system prompt) ===')
print(f'Without sharing: {without_sharing} blocks for prefixes')
print(f'With sharing:    {with_sharing} blocks (1 physical copy, ref_count=8)')
print(f'Saved:           {saved_blocks} blocks ({saved_blocks * 16:,} tokens)')
print(f'Reduction:       {saved_blocks / without_sharing:.0%}')
print(f'\nTotal blocks in use: {cache2.alloc.num_used}, free: {cache2.alloc.num_free}')

## 5. Copy-on-Write for Beam Search

Beam expansion forks sequences by sharing blocks. When a beam diverges
(writes to a shared block), CoW triggers a physical copy.

In [ ]:
class BeamSearchSimulator:
    def __init__(self, num_blocks=512, block_size=16):
        self.alloc = BlockAllocator(num_blocks, block_size)
        self.bs = block_size
        self.beams: Dict[int, List[int]] = {}
        self.cow_copies = 0

    def init_prompt(self, tokens: int):
        n = (tokens + self.bs - 1) // self.bs
        self.beams[0] = [self.alloc.allocate() for _ in range(n)]
        for bid in self.beams[0]:
            self.alloc.blocks[bid].tokens_filled = self.bs

    def fork(self, parent: int, child: int):
        for bid in self.beams[parent]:
            self.alloc.ref(bid)
        self.beams[child] = self.beams[parent].copy()

    def append_token(self, beam_id: int):
        blocks = self.beams[beam_id]
        last = blocks[-1]
        # CoW: if shared, copy before write
        if self.alloc.blocks[last].ref_count > 1:
            new_bid = self.alloc.cow_copy(last)
            blocks[-1] = new_bid
            self.cow_copies += 1
        self.alloc.blocks[blocks[-1]].tokens_filled += 1
        if self.alloc.blocks[blocks[-1]].tokens_filled >= self.bs:
            blocks.append(self.alloc.allocate())

# Simulate: 4 beams, 128-token prompt, generate 64 tokens
sim = BeamSearchSimulator(num_blocks=512, block_size=16)
sim.init_prompt(128)
for b in range(1, 4):
    sim.fork(0, b)

blocks_after_fork = sim.alloc.num_used
naive_after_fork = 4 * ((128 + 15) // 16)  # full copy per beam

for step in range(64):
    for beam_id in range(4):
        sim.append_token(beam_id)

blocks_final = sim.alloc.num_used
naive_final = 4 * ((128 + 64 + 15) // 16)

print('=== Copy-on-Write Beam Search (4 beams, 64 gen tokens) ===')
print(f'After fork:  {blocks_after_fork} blocks used (naive: {naive_after_fork})')
print(f'After gen:   {blocks_final} blocks used (naive: {naive_final})')
print(f'CoW copies:  {sim.cow_copies}')
print(f'Memory saved: {(1 - blocks_final/naive_final):.1%} vs naive duplication')

## 6. Throughput Under Load: Paged vs Contiguous

In [ ]:
def simulate_serving(n_requests, mode, total_blocks=4096, block_size=16, max_seq=2048):
    np.random.seed(0)
    used, served, rejected = 0, 0, 0
    for _ in range(n_requests):
        seq_len = np.random.randint(64, max_seq)
        needed = (max_seq // block_size) if mode == 'contiguous' else ((seq_len + block_size - 1) // block_size)
        if used + needed <= total_blocks:
            used += needed
            served += 1
        else:
            rejected += 1
    return served, rejected

print(f'{"Requests":>10} | {"Contiguous":>10} | {"Paged":>10} | {"Improvement":>12}')
print('-' * 52)
for n in [50, 100, 200, 500]:
    cs, _ = simulate_serving(n, 'contiguous')
    ps, _ = simulate_serving(n, 'paged')
    print(f'{n:>10} | {cs:>10} | {ps:>10} | {ps-cs:>+10} ({(ps-cs)/max(cs,1):.0%})')

## 7. Real Model Memory Analysis

Using the shared `kv_efficiency` utility to compute actual memory numbers.

In [ ]:
# Llama-2 70B style: 80 layers, 8 KV heads (GQA), head_dim=128
configs = [
    ('Llama-7B',  32, 32, 128),
    ('Llama-13B', 40, 40, 128),
    ('Llama-70B', 80,  8, 128),
]

batch, context = 32, 2048
block_size = 16

print(f'=== KV Cache Memory: Contiguous vs Paged (batch={batch}, ctx={context}) ===')
print(f'{"Model":<12} | {"KV (GiB)":>9} | {"Contig Waste":>12} | {"Paged Waste":>11} | {"Saving":>7}')
print('-' * 65)

for name, layers, kv_heads, hdim in configs:
    kv_gib = kv_cache_size_gib(layers, kv_heads, hdim, context, batch)
    # Contiguous wastes ~50% on average (uniform [32,2048] -> mean ~1040, max 2048)
    contig_waste = kv_gib * 0.49  # avg(max-actual)/max for uniform dist
    # Paged wastes at most (block_size-1)/avg_seq per sequence
    avg_seq = 1040
    paged_waste_frac = (block_size - 1) / (2 * avg_seq)  # ~0.7%
    paged_waste = kv_gib * paged_waste_frac
    saving = contig_waste - paged_waste
    print(f'{name:<12} | {kv_gib:>8.2f}  | {contig_waste:>10.2f} GiB | {paged_waste:>9.3f} GiB | {saving:>5.2f} GiB')

In [ ]:
# --- GPU context: how much of VRAM does KV cache consume? ---
try:
    gpu = get_gpu_info()
    gpu_name = gpu.name
    vram = gpu.vram_gb
except Exception:
    gpu_name, vram = 'A100-80GB', 80.0  # fallback

print(f'\n=== KV Cache as % of VRAM ({gpu_name}, {vram} GB) ===')
for name, layers, kv_heads, hdim in configs:
    kv_gib = kv_cache_size_gib(layers, kv_heads, hdim, context, batch)
    pct = kv_gib / vram * 100
    max_batch_paged = int(batch * (vram * 0.7) / kv_gib)  # 70% VRAM budget for KV
    print(f'  {name}: {kv_gib:.2f} GiB = {pct:.1f}% VRAM | max batch (paged, 70% budget): {max_batch_paged}')

## Key Takeaways

| Mechanism | What it eliminates | Typical saving |
|---|---|---|
| Block allocation | External fragmentation | 2-4x more concurrent seqs |
| Small block size (16) | Internal frag bounded to 15 tokens/seq | <1% waste |
| Prefix sharing | Redundant system prompt copies | 7/8 = 87.5% for 8 shared reqs |
| Copy-on-Write | Eager beam duplication | ~20-40% during beam search |

**vLLM implementation notes:**
- Block table lives on GPU; attention kernel indexes `kv[block_table[logical_idx]][offset]`
- Preemption: swap LRU sequence blocks to CPU when OOM, resume later
- Default block_size=16 balances CUDA warp efficiency vs fragmentation